# EXP-10: BPTI–Trypsin Activation Energy
**PMF barrier height = activation energy for dissociation**

- **Expected:** Ea = 10.5 kcal/mol (Quast 1974)
- **PASS:** [4.6, 16.4] | **MARGINAL:** [2.0, 20.0] | **FAIL:** outside
- **Depends on:** EXP-04 umbrella timeseries on Drive

In [ ]:
# Install OpenMM + all scientific dependencies (pip — OpenCL platform)
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git

# mpiplus stub — not on PyPI; minimal shim for imports
import os as _os, sys as _sys
_sp = _os.path.join(_sys.prefix, 'lib',
    f'python{_sys.version_info.major}.{_sys.version_info.minor}',
    'site-packages', 'mpiplus')
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]

!pip install -q scipy matplotlib pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)
import sys, os, shutil, json, zipfile, logging
from pathlib import Path
import numpy as np
from scipy.signal import argrelextrema

PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists()
sys.path.insert(0, str(PIPELINE_ROOT))

EXP_ID = 'EXP-10'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
for d in ['analysis','figures']: (WORK_DIR/d).mkdir(parents=True, exist_ok=True)

EXP04 = Path('/content/drive/MyDrive/v3_gpu_results/EXP-04')
assert EXP04.exists(), 'Run EXP-04 first'

from src.config import UmbrellaConfig, WHAMConfig, MBARConfig, KCAL_TO_KJ
from src.simulate.umbrella import generate_window_centers
from src.analyze.wham import solve_wham, bootstrap_pmf_uncertainty
from src.analyze.mbar import solve_mbar
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
logging.basicConfig(level=logging.INFO)
print('Imports OK')

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

In [ ]:
# Load EXP-04 PMF data
pmf_file = list(EXP04.glob('**/pmf_data.npz'))
assert pmf_file, 'EXP-04 pmf_data.npz not found'
data = np.load(str(pmf_file[0]))
xi = data['xi']
pmf_kcal = data['pmf_kcal']
fm = np.isfinite(pmf_kcal)

# Also try loading umbrella timeseries for WHAM reanalysis + MBAR
us_npz = list(EXP04.glob('**/umbrella_timeseries.npz'))
print(f'PMF: {len(xi)} bins')

In [ ]:
# Extract barrier height
idx_min = np.argmin(pmf_kcal[fm])
idx_min_g = np.where(fm)[0][idx_min]
xi_min = xi[idx_min_g]

search = pmf_kcal[idx_min_g:]
maxima = argrelextrema(search, np.greater, order=5)[0]
if len(maxima) > 0:
    idx_ts = maxima[0] + idx_min_g
else:
    idx_ts = idx_min_g + len(search) // 2

xi_ts = xi[idx_ts]
barrier_wham = pmf_kcal[idx_ts] - pmf_kcal[idx_min_g]

# Bootstrap uncertainty from saved PMF std if available
pmf_std = np.zeros_like(pmf_kcal)
if 'pmf_std' in data.files:
    pmf_std = data['pmf_std']
barrier_std = np.sqrt(pmf_std[idx_ts]**2 + pmf_std[idx_min_g]**2) if np.any(pmf_std > 0) else barrier_wham * 0.15

print(f'\u03be_min = {xi_min:.2f} nm, \u03be\u2021 = {xi_ts:.2f} nm')
print(f'Barrier (WHAM): \u0394W\u2021 = {barrier_wham:.2f} \u00b1 {barrier_std:.2f} kcal/mol')

In [ ]:
# Block convergence: split PMF data into first/second half if umbrella data available
barrier_first = barrier_wham  # defaults
barrier_second = barrier_wham
block_agreement = 0.0

if us_npz:
    us_data = np.load(str(us_npz[0]), allow_pickle=True)
    if 'window_centers' in us_data:
        wc = us_data['window_centers']
        n_win = len(wc)
        # Reconstruct timeseries lists
        xi_ts_list = []
        for i in range(n_win):
            key = f'xi_window_{i:03d}'
            if key in us_data:
                xi_ts_list.append(us_data[key])
        if len(xi_ts_list) == n_win:
            ks = np.full(n_win, 1000.0)  # spring constant
            half_n = min(len(ts) for ts in xi_ts_list) // 2
            first = [ts[:half_n] for ts in xi_ts_list]
            second = [ts[half_n:2*half_n] for ts in xi_ts_list]
            wham_cfg = WHAMConfig()
            try:
                r1 = solve_wham(first, wc, ks, 310.0, wham_cfg)
                r2 = solve_wham(second, wc, ks, 310.0, wham_cfg)
                p1k = r1['pmf_kcal_mol']; p2k = r2['pmf_kcal_mol']
                barrier_first = p1k[idx_ts] - np.min(p1k[fm])
                barrier_second = p2k[idx_ts] - np.min(p2k[fm])
                block_agreement = abs(barrier_first - barrier_second)
                print(f'Block convergence: first={barrier_first:.2f}, second={barrier_second:.2f}, diff={block_agreement:.2f}')
            except Exception as e:
                print(f'Block analysis skipped: {e}')

In [ ]:
# Classification
ea_exp = 10.5
if barrier_wham < 1.0:
    verdict = 'FAIL'  # no clear barrier
elif 4.6 <= barrier_wham <= 16.4:
    verdict = 'PASS'
elif 2.0 <= barrier_wham <= 20.0:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'Ea (PMF barrier) = {barrier_wham:.2f} kcal/mol vs Exp {ea_exp}')
print(f'Verdict: {verdict}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Fig 1: PMF with barrier annotation
ax = axes[0]
ax.plot(xi[fm], pmf_kcal[fm], 'b-', lw=2, label='WHAM')
if np.any(pmf_std > 0):
    ax.fill_between(xi[fm], (pmf_kcal-1.96*pmf_std)[fm], (pmf_kcal+1.96*pmf_std)[fm],
                    alpha=0.15, color='blue')
ax.annotate('', xy=(xi_ts, pmf_kcal[idx_ts]), xytext=(xi_ts, pmf_kcal[idx_min_g]),
            arrowprops=dict(arrowstyle='<->', color='darkred', lw=2))
ax.text(xi_ts+0.1, (pmf_kcal[idx_ts]+pmf_kcal[idx_min_g])/2,
        f'\u0394W\u2021={barrier_wham:.1f}', fontsize=12, color='darkred')
ax.axhline(y=ea_exp+pmf_kcal[idx_min_g], color='gold', ls=':', lw=2, label=f'Exp Ea={ea_exp}')
ax.set_xlabel('\u03be (nm)'); ax.set_ylabel('PMF (kcal/mol)'); ax.set_title('Activation Energy'); ax.legend()

# Fig 2: Block convergence
ax = axes[1]
labels = ['Full PMF', 'First half', 'Second half', f'Exp ({ea_exp})']
vals = [barrier_wham, barrier_first, barrier_second, ea_exp]
colors = ['steelblue', 'lightblue', 'lightblue', 'gold']
ax.bar(labels, vals, color=colors, edgecolor='black')
ax.axhspan(4.6, 16.4, alpha=0.1, color='green', label='PASS range')
ax.set_ylabel('Barrier (kcal/mol)'); ax.set_title('Convergence'); ax.legend()

fig.suptitle(f'EXP-10: Activation Energy \u2014 {verdict}', fontsize=14)
fig.tight_layout(); fig.savefig(WORK_DIR/'figures'/'results.png', dpi=150); plt.close(fig)

In [ ]:
results = {
    'experiment_id': EXP_ID, 'barrier_wham_kcal': float(barrier_wham),
    'barrier_std_kcal': float(barrier_std),
    'barrier_first_kcal': float(barrier_first), 'barrier_second_kcal': float(barrier_second),
    'block_agreement_kcal': float(block_agreement),
    'ea_exp_kcal': ea_exp, 'xi_min_nm': float(xi_min), 'xi_ts_nm': float(xi_ts),
    'verdict': verdict,
}
with open(WORK_DIR/'results.json', 'w') as f: json.dump(results, f, indent=2)

for item in WORK_DIR.rglob('*'):
    if item.is_file():
        dest = DRIVE_OUTPUT / item.relative_to(WORK_DIR)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)
# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file(): zf.write(item, f'{EXP_ID}/{item.relative_to(WORK_DIR)}')
files.download(str(zip_path))